# M1M3 z-Gradient Survey (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-07-07
**Last Modified:** 2026-07-07
**Status:** In Progress
**Keywords:** M1M3, thermal gradient, z_gradient, EFD, thermocouples, bounce test

## Description

Survey the M1M3 vertical thermal gradient (`z_gradient`) across all `science` and
`acq` exposures over a date range.  `z_gradient` is **not** in the ConsDB — it is
derived from the EFD M1M3 thermocouple telemetry via
`lsst.ts.m1m3.utils.ThermocoupleAnalysis` (the same `xyz_r_gradients` the AOS
`ts_intrinsic_wavefront` pipeline uses to enrich its visit tables).

For speed over a multi-month range the EFD is loaded in **multi-day chunks** at a
**coarse cadence** (default 5 min — z_gradient varies on ~10-min scales), and the
gradient time series is interpolated to each exposure's start time.  The result is
cached to parquet and the fetch is skipped if the cache already exists.

Key functionality:
1. If the cache is absent: list all `science`/`acq` exposures over [START, END]
   (one Butler query) and fill `z_gradient` from the EFD in `CHUNK_DAYS` windows at
   `TIME_BIN_SEC` cadence (tqdm per-chunk); cache to parquet.
2. Violin plot of `z_gradient` per `day_obs` on a **calendar** axis, so nights with
   no science/acq exposures appear as gaps.

**Output:** `z_gradient_science_acq.parquet` + a per-night violin figure.

**References:** `ts_intrinsic_wavefront/.../intrinsics_lib.py` (`get_m1m3_data`,
`ThermocoupleAnalysis.xyz_r_gradients`); ConsDB `cdb_lsstcam` schema (z_gradient absent).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-07 | Aaron Roodman | Initial version — science/acq z_gradient survey, chunked/coarse EFD loads |
| 2026-07-07 | Aaron Roodman | Cache-skip if parquet exists; violin-only plot on a calendar axis (gaps for empty nights) |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Fetch z_gradient per exposure](#fetch)
4. [Violin plot per day_obs](#plot)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — all configurable values collected here
# ============================================================
from datetime import date

START = "2026-03-15"                 # first calendar day (inclusive)
END = date.today().isoformat()       # last calendar day (inclusive) — default: today
KEEP_TYPES = ("science", "acq")      # exposure.observation_type values to keep
BUTLER_REPO = "/repo/main"
EFD_NAME = "usdf_efd"                 # EFD instance for this site
CHUNK_DAYS = 7                        # EFD load window (bigger = fewer round-trips)
TIME_BIN_SEC = 300                   # EFD gradient cadence (s); z_gradient is slow
OUT = "z_gradient_science_acq.parquet"   # cache path (cwd); delete to force a rebuild
FIG = "z_gradient_science_acq.png"

<a id='setup'></a>
## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm                    # plain text bar (no ipywidgets dependency)
from astropy.time import Time

from lsst.daf.butler import Butler
from lsst_efd_client import EfdClient
from lsst.ts.m1m3.utils import ThermocoupleAnalysis

butler = Butler(BUTLER_REPO, instrument="LSSTCam")
# Construct EfdClient directly with output_mode="dataframe"; makeEfdClient() can fail
# with "Invalid output format" on some summit_utils / lsst_efd_client combinations.
efd = EfdClient(EFD_NAME, output_mode="dataframe")

<a id='fetch'></a>
## Fetch z_gradient per exposure

Skips the fetch if `OUT` already exists (delete it to rebuild). EFD loaded in multi-day chunks at coarse cadence (tune `CHUNK_DAYS` / `TIME_BIN_SEC`); `await` works at notebook top level.

In [ ]:
if Path(OUT).exists():
    print(f"cache exists -> skipping fetch: {OUT}  (delete it to rebuild)")
else:
    # 1) all science/acq exposures in the range, in ONE Butler query
    start_day = int(START.replace("-", "")); end_day = int(END.replace("-", ""))
    recs = butler.registry.queryDimensionRecords(
        "exposure", instrument="LSSTCam",
        where=f"exposure.day_obs >= {start_day} AND exposure.day_obs <= {end_day}")
    exp = [(r.day_obs, r.seq_num, r.observation_type, r.timespan.begin)
           for r in recs if r.observation_type in KEEP_TYPES]
    df = pd.DataFrame(exp, columns=["day_obs", "seq_num", "observation_type", "t_begin"])
    df = df.sort_values("t_begin").reset_index(drop=True)
    df["t_ns"] = (Time(list(df["t_begin"])).unix * 1e9).astype("int64")   # UTC ns
    df["z_gradient"] = np.nan
    print(f"{len(df)} science/acq exposures over {df.day_obs.nunique()} nights")

    # 2) EFD M1M3 gradients in CHUNK_DAYS windows at TIME_BIN_SEC cadence -> interp
    end_ts = pd.Timestamp(END) + pd.Timedelta(days=1)
    edges = list(pd.date_range(START, end_ts, freq=f"{CHUNK_DAYS}D"))
    if not edges or edges[-1] < end_ts:          # always close the final window
        edges.append(end_ts)
    for i in tqdm(range(len(edges) - 1), desc=f"{CHUNK_DAYS}-day EFD chunks"):
        c0, c1 = edges[i], edges[i + 1]
        sel = (df.t_ns >= c0.value) & (df.t_ns < c1.value)
        if not sel.any():
            continue
        try:
            tca = ThermocoupleAnalysis(efd)
            await tca.load(Time(c0.isoformat(), scale="utc"),
                           Time(c1.isoformat(), scale="utc"), time_bin=TIME_BIN_SEC)
            g = tca.xyz_r_gradients
            gt = pd.to_datetime(g.index, utc=True).astype("int64").to_numpy()
            gz = np.asarray(g["z_gradient"], float)
            o = np.argsort(gt)
            df.loc[sel, "z_gradient"] = np.interp(df.loc[sel, "t_ns"].to_numpy(),
                                                  gt[o], gz[o])
        except Exception as e:
            tqdm.write(f"  {c0.date()}..{c1.date()}: failed ({type(e).__name__}: {e})")

    df.drop(columns=["t_begin", "t_ns"]).to_parquet(OUT)
    print(f"z_gradient filled for {int(np.isfinite(df.z_gradient).sum())}/{len(df)} "
          f"exposures -> {OUT}")

<a id='plot'></a>
## Violin plot per day_obs

In [ ]:
df = pd.read_parquet(OUT)
df = df[np.isfinite(df.z_gradient)]

# calendar axis: position each night at its real day offset so missing nights = gaps
df["date"] = pd.to_datetime(df.day_obs.astype(str), format="%Y%m%d")
d0 = df.date.min()
days = np.sort(df.day_obs.unique())
offs = [(pd.to_datetime(str(d), format="%Y%m%d") - d0).days for d in days]
data = []
for d in days:
    z = df.z_gradient[df.day_obs == d].values
    data.append(z if z.size >= 2 else (np.full(2, z[0]) if z.size == 1
                                       else np.array([np.nan, np.nan])))

span = (offs[-1] - offs[0]) + 1
fig, ax = plt.subplots(figsize=(max(10, 0.16 * span), 5), constrained_layout=True)
vp = ax.violinplot(data, positions=offs, widths=0.9, showmedians=True, showextrema=False)
for b in vp["bodies"]:
    b.set_facecolor("steelblue"); b.set_alpha(0.55)
ax.axhline(0, color="k", lw=0.5, ls="--", alpha=0.5)
xt = np.arange(0, span, max(1, span // 15))
ax.set_xticks(xt)
ax.set_xticklabels([(d0 + pd.Timedelta(days=int(t))).strftime("%m-%d") for t in xt],
                   rotation=90, fontsize=7)
ax.set_xlabel("day_obs (calendar; gaps = nights with no science/acq exposures)")
ax.set_ylabel("M1M3 z_gradient")
ax.set_title(f"M1M3 z_gradient per night  "
             f"({df.day_obs.min()}-{df.day_obs.max()}, {len(days)} nights, N={len(df)})")
ax.grid(axis="y", alpha=0.3)
fig.savefig(FIG, dpi=140)
plt.show()